<a href="https://colab.research.google.com/github/Balachandar-Ganesan/DeepLearning/blob/main/300_BuildYourOwnTensor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

import numpy as np
NDArray = np.ndarray
def _ensure_tensor(val):
    return val if isinstance(val, Tensor) else Tensor(val,
     requires_grad=False)
class Tensor:
    def __init__(self, data, *, device=None, dtype="float32",
     requires_grad=False):
        if isinstance(data, Tensor):
            if dtype is None:
                dtype = data.dtype
            self.data = data.numpy().astype(dtype)
        elif isinstance(data, np.ndarray):
            self.data = data.astype(dtype if dtype is not None else data.dtype)
        else:
            self.data = np.array(data, dtype=dtype if dtype is not None
             else "float32")
        self.grad = None
        self.requires_grad = requires_grad
        self._op = None
        self._inputs = []
        self._device = device if device else  "cpu"
    def numpy(self):
        return self.data.copy()
    def detach(self):
        return Tensor(self.data, requires_grad=False, dtype=str(self.dtype))
    @property
    def shape(self):
        return self.data.shape
    @property
    def dtype(self):
        return self.data.dtype
    @property
    def ndim(self):
        return self.data.ndim
    @property
    def size(self):
        return self.data.size
    @property
    def device(self):
        return self._device
    @property
    def T(self):
        return self.transpose()
    def __repr__(self):
        return f"Tensor({self.data}, requires_grad={self.requires_grad})"
    def __str__(self):
        return str(self.data)
    @classmethod
    def rand(cls, *shape, low=0.0, high=1.0, dtype="float32",
        requires_grad=True):
        array = np.random.rand(*shape) * (high - low) + low
        return cls(array.astype(dtype), requires_grad=requires_grad)
    @classmethod
    def randn(cls, *shape, mean=0.0, std=1.0, dtype="float32",
         requires_grad=True):
        array = np.random.randn(*shape) * std + mean
        return cls(array.astype(dtype), requires_grad=requires_grad)
    @classmethod
    def constant(cls, *shape, c=1.0, dtype="float32", requires_grad=True):
        array = np.ones(*shape) * c
        return cls(array.astype(dtype), requires_grad=requires_grad)
    @classmethod
    def ones(cls, *shape, dtype="float32", requires_grad=True):
        return cls.constant(*shape, c=1.0, dtype=dtype,
        requires_grad=requires_grad)
    @classmethod
    def zeros(cls, *shape, dtype="float32", requires_grad=True):
        return cls.constant(*shape, c=0.0, dtype=dtype,
         requires_grad=requires_grad)
    @classmethod
    def randb(cls, *shape, p=0.5, dtype="float32", requires_grad=True):
        array =np.random.rand(*shape) <=p
        return cls(array,dtype=dtype, requires_grad=requires_grad )
    @classmethod
    def empty(cls, *shape, dtype="float32", requires_grad=True):
        array =np.empty(shape,dtype=dtype)
        return cls(array, requires_grad=requires_grad)

    @classmethod
    def one_hot(cls,indices,num_classes,device=None, dtype="float32",
         requires_grad=True):
        one_hot_array = np.eye(num_classes,dtype=dtype)[np.array(
            indices.data,dtype=int)]
        return cls(one_hot_array,device=device, dtype=dtype,
            requires_grad=requires_grad)


In [9]:

from typing import Tuple
import numpy as np
class Function:
    def __call__(self, *args):
        raise NotImplementedError()
    def forward(self, *args):
        """Computes the forward pass of the operation.
        Args:
            *args: One or more NumPy arrays
        """
        raise NotImplementedError()
    def backward(self, out_grad, node):
        """Calculates backward pass (gradients)
        Args:
            out_grad: upstream gradient flowing from output to input
            node: Value object holding inputs from forward pass
        """
        pass

In [12]:
class Pow(Function):
    def forward(self, a, b):
        return np.power(a, b)
    def backward(self, out_grad, node):
        a, b = node._inputs
        grad_a = multiply(multiply(out_grad, b), power(a, add_scalar(b, -1)))
        grad_b = multiply(multiply(out_grad, power(a, b)), log(a))
        return grad_a, grad_b
def power(a, b):
    return Pow()(a, b)
class PowerScalar(Function):
    def __init__(self, scalar: int):
        self.scalar = scalar
    def forward(self, a: NDArray) -> NDArray:
        return np.power(a ,self.scalar)
    def backward(self, out_grad, node):
        inp = node._inputs[0]
        grad = multiply(out_grad, multiply(Tensor(self.scalar),
         power_scalar(inp, self.scalar - 1)))
        return grad
def power_scalar(a, scalar):
    return PowerScalar(scalar)(a)
class Div(Function):
    def forward(self, a, b):
        return a/b
    def backward(self, out_grad, node):
        x,y = node._inputs
        grad_x = divide(out_grad, y)
        grad_y = multiply(negate(out_grad), divide(x, multiply(y, y)))
        return grad_x, grad_y
def divide(a, b):
    return Div()(a, b)
class DivScalar(Function):
    def __init__(self, scalar):
        self.scalar = scalar
    def forward(self, a):
        return np.array(a / self.scalar, dtype=a.dtype)
    def backward(self, out_grad, node):
        return  out_grad/self.scalar
def divide_scalar(a, scalar):
    return DivScalar(scalar)(a)

In [13]:
class Negate(Function):
    def forward(self, a):
        return -a
    def backward(self, out_grad, node):
        return negate(out_grad)
def negate(a):
    return Negate()(a)
class Log(Function):
    def forward(self, a):
        return np.log(a)
    def backward(self, out_grad, node):
        # f'(x) = 1/x
        inp = node._inputs[0]
        return divide(out_grad, inp)
def log(a):
    return Log()(a)
class Exp(Function):
    def forward(self, a):
        return np.exp(a)
    def backward(self, out_grad, node):
        # We already calculated exp(x) in forward, it's stored in 'node'.
        return multiply(out_grad, node)
def exp(a):
    return Exp()(a)
class Sqrt(Function):
    def forward(self, a):
        return np.sqrt(a)
    def backward(self, out_grad, node):
        # f'(x) = 1 / (2 * sqrt(x))
        # Again, 'node' IS sqrt(x). We use it directly.
        two = Tensor(2.0)
        return divide(out_grad, multiply(two, node))
def sqrt(a):
    return Sqrt()(a)

In [14]:
class ReLU(Function):
    def forward(self, a):
        ### BEGIN YOUR SOLUTION
        a = a * (a>0)
        return a
    def backward(self, out_grad, node):
        inp = node._inputs[0]
        mask = Tensor((inp.data > 0).astype("float32"), requires_grad=False)
        return multiply(out_grad, mask)

def relu(a):
    return ReLU()(a)

class Sigmoid(Function):
    def forward(self, a):
        out = 1/(1+np.exp(-a))
        return out
    def backward(self, out_grad, node):
        one = Tensor(1.0, requires_grad=False)
        local_grad = multiply(node, add(one, negate(node)))
        return multiply(out_grad, local_grad)

def sigmoid(x):
    return Sigmoid()(x)

class Tanh(Function):
    def forward(self,a):
        return np.tanh(a)
    def backward(self, out_grad, node):
        one = Tensor(1.0, requires_grad=False)
        squared = multiply(node, node)
        local_grad = add(one, negate(squared))
        return multiply(out_grad, local_grad)

def tanh(x):
    return Tanh()(x)

In [15]:
class Reshape(Function):
    def __init__(self, shape):
        self.shape = shape
    def forward(self, a):
        return np.reshape(a,self.shape)
    def backward(self, out_grad, node):
        a = node._inputs[0]
        return reshape(out_grad, a.shape)

def reshape(a, shape):
    return Reshape(shape)(a)

In [16]:

class Transpose(Function):
    def __init__(self, axes: Optional[tuple] = None):
        self.axes = axes
    def forward(self, a):
        if self.axes is None:
            return np.swapaxes(a, -1, -2)

        ndim = a.ndim
        #handling -ve axes
        axes = tuple(ax if ax >= 0 else ndim + ax for ax in self.axes)
        if len(axes) == 2:
            full_axes = list(range(ndim))
            i, j = axes
            full_axes[i], full_axes[j] = full_axes[j], full_axes[i]
            self.full_axes = tuple(full_axes)
        else:
            self.full_axes = axes

        return np.transpose(a, self.full_axes)
    def backward(self, out_grad, node):
        if self.axes is None:
            return transpose(out_grad)
        inverse_axes = np.argsort(self.axes)
        return transpose(out_grad, tuple(inverse_axes))

def transpose(a, axes=None):
    return Transpose(axes)(a)


NameError: name 'Optional' is not defined

In [17]:
class Summation(Function):
    def __init__(self, axes: Optional[tuple] = None):
        self.axes = axes

    def forward(self, a):
        return np.sum(a, axis=self.axes)

    def backward(self, out_grad, node):
        a = node._inputs[0]

        original_shape = a.shape

        if self.axes is None:
            intermediate_shape = (1,) * len(original_shape)
        else:
            axes = self.axes if isinstance(self.axes, (list, tuple))
                     else (self.axes,)
            axes= [ax if ax >= 0 else ax + len(original_shape) for ax in axes]
            intermediate_shape = list(out_grad.shape)

            #inserting 1's where the axis was vanished.
            for ax in sorted(axes):
                intermediate_shape.insert(ax, 1)
        #reshape
        reshaped_grad = reshape(out_grad, tuple(intermediate_shape))
        ones = np.ones(original_shape)
        ones = Tensor(ones)
        #broadcast or multiply by ones
        return reshaped_grad * ones

SyntaxError: expected 'else' after 'if' expression (2012649947.py, line 16)

In [18]:

class BroadcastTo(Function):
    def __init__(self, shape):
        self.shape = shape

    def forward(self, a):
        return np.broadcast_to(a, self.shape)

    def backward(self, out_grad, node):
        a = node._inputs[0]
        original_shape = a.shape
        converted_shape = out_grad.shape

        #Un-Prepending
        changed_shape = len(converted_shape) -len(original_shape)
        grad =out_grad
        for _ in range(changed_shape):
            grad = summation(grad, axes=0)

        # Un-strectching
        for i, (orig_dim, new_dim) in enumerate(zip(original_shape, grad.shape)):
            if orig_dim ==1 and new_dim > 1 :
                grad = summation(grad, axes=i)
                new_shape = list(grad.shape)
                # numpy sometimes does (n,) instead of (n,1).
                # We insert (1) for reshaping.
                new_shape.insert(i, 1)
                grad = reshape(grad, tuple(new_shape))

        return grad

In [19]:
class MatMul(Function):
    def forward(self, a, b):
        return np.matmul(a, b)

    def backward(self, out_grad, node):
        a, b = node._inputs

        if len(out_grad.shape) == 0:
            out_grad = out_grad.broadcast_to(node.shape)

        grad_a = matmul(out_grad, transpose(b, axes=(-1, -2)))
        grad_b = matmul(transpose(a, axes=(-1, -2)), out_grad)

        while len(grad_a.shape) > len(a.shape):
            grad_a = summation(grad_a, axes=0)
        while len(grad_b.shape) > len(b.shape):
            grad_b = summation(grad_b, axes=0)

        grad_a = grad_a.reshape(a.shape)
        grad_b = grad_b.reshape(b.shape)
        return grad_a, grad_b
def matmul(a, b):
    return MatMul()(a, b)